# 🛡️ Dry-Run Harness: Preview an Action, Execute Only on Approval

## Learning Objectives
In this notebook, you will learn:
1. **The dry-run harness** - Separate *proposing* an action from *executing* it, with a preview in between
2. **A tool with a dry-run mode** - One function that either describes what it would do or does it
3. **Pausing a graph for a human** - `interrupt()` in a review node, a checkpointer, and resuming with `Command(resume=...)`
4. **Approve and reject, without typing** - A scripted reviewer drives the pause/resume loop so the whole notebook runs unattended
5. **Failing closed** - Unclear answers count as rejections, and an approval only covers the exact action that was previewed
6. **Reviewing interactively** - The same loop with a real person at the keyboard

## Where this fits
`modules/04_langchain_agents_and_middleware/notebooks/04_middleware.ipynb` used `HumanInTheLoopMiddleware` to pause a `create_agent` agent before a risky tool call. That middleware is built on the mechanism this notebook uses directly: a LangGraph `interrupt()` plus a checkpointer. Building the gate by hand lets us add the two things the middleware does not give you out of the box: a **dry-run preview** of the effect, and a **fingerprint** that ties the approval to exactly what was previewed. `modules/12_advanced_cognitive_patterns/notebooks/05_reflexive_metacognitive.ipynb` decided *when* to escalate. This notebook shows *how* the human decision is collected and enforced.

## Prerequisites
- LangGraph state, nodes and conditional edges - `modules/03_react_agents/`
- Interrupts and `Command(resume=...)` at the middleware level - `modules/04_langchain_agents_and_middleware/notebooks/04_middleware.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Software practice ("dry-run" flags in `rsync`, `apt`, `terraform plan`); agentic version: pre-execution human approval |

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The social-media agent that drafts posts
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📖 Part 1: The Dry-Run Harness

The core principle is simple: **never run an agent's action in a live environment without first knowing exactly what it's going to do.** The agent first executes its plan in a `dry_run` mode, which doesn't change the real world but produces a clear preview. That preview goes to a human (or an automated checker), and only an explicit approval allows the live execution.

### High-level Workflow
1. **Propose:** the agent drafts the action (here: an LLM writes a social-media post).
2. **Dry run:** the tool is called with `dry_run=True`. It returns exactly what it *would* do, and changes nothing.
3. **Review:** the graph **pauses** and hands the preview to a reviewer.
4. **Decide:** the reviewer answers `approve` or `reject`. Anything else counts as `reject`.
5. **Execute or cancel:** only on approval is the tool called again with `dry_run=False`, and only for the action that was previewed.

### When to Use / Applications
*   **Irreversible or visible actions:** sending messages, publishing, payments, deleting data, changing production systems.
*   **Debugging agents:** see exactly what an agent would do without having to undo real changes.
*   **Compliance:** an auditable record of who approved which action, and when.

### Strengths & Weaknesses
*   **Strengths:** the effect is previewed rather than guessed; a human keeps the final say; every decision is logged.
*   **Weaknesses:** it adds latency (a human in the loop is not real-time), and the tool must support a dry-run mode that shows what the live call will do.

---
## 🔧 Part 2: A Tool with a Dry-Run Mode

`SocialMediaAPI.publish_post` has one code path for building the post and two outcomes: with `dry_run=True` it returns a preview and a **fingerprint** (a SHA-256 hash of the exact text). With `dry_run=False` it appends the post to `feed`, our stand-in for the real platform. Because the feed is a list we can inspect, every claim below ("nothing was published") is checked with an `assert`.

In [ ]:
# ============================================================================
# DRY-RUN-CAPABLE TOOL: preview + fingerprint, or a real (mock) publish
# ============================================================================
import datetime
import hashlib
from typing import List

from pydantic import BaseModel, Field


class SocialMediaPost(BaseModel):
    content: str = Field(description="The text of the post, WITHOUT hashtags")
    hashtags: List[str] = Field(description="Relevant hashtags, without the '#'")


def render(post: SocialMediaPost) -> str:
    return f"{post.content}\n\n" + " ".join(f"#{h}" for h in post.hashtags)


def fingerprint(post: SocialMediaPost) -> str:
    return hashlib.sha256(render(post).encode()).hexdigest()[:12]


class SocialMediaAPI:
    """A mock social-media API that supports a dry-run mode."""

    def __init__(self):
        self.feed = []  # what has really been published

    def publish_post(self, post: SocialMediaPost, dry_run: bool = True) -> dict:
        text = render(post)
        if dry_run:
            return {"status": "DRY_RUN", "preview": text, "fingerprint": fingerprint(post)}
        post_id = f"post_{len(self.feed) + 1}"
        self.feed.append({"id": post_id, "text": text, "at": datetime.datetime.now().isoformat(timespec="seconds")})
        return {"status": "PUBLISHED", "post_id": post_id}


api = SocialMediaAPI()
demo = SocialMediaPost(content="Hello world.", hashtags=["demo"])
print(api.publish_post(demo, dry_run=True))
assert api.feed == [], "a dry run must not publish"
print("✅ Dry run returned a preview and published nothing")

---
## ⏸️ Part 3: Pausing the Graph for a Human

The review node calls `interrupt(payload)`. The first time it runs, LangGraph **stops the graph**, saves its state in the checkpointer, and returns the payload to the caller under `"__interrupt__"`. When the caller invokes the graph again with `Command(resume=decision)` on the same `thread_id`, the review node runs again from the top, and this time `interrupt()` **returns** `decision`.

Three rules make the gate safe:
- **Fail closed:** `normalize_decision` turns anything that is not a clear `approve` into `reject`.
- **Approve what was previewed:** the approval stores the preview's fingerprint. `publish` recomputes the fingerprint of the post it is about to publish and refuses if they differ.
- **Log everything:** every decision is appended to an audit log with the reviewer, the reason and the fingerprint.

Because the review node re-runs on resume, everything before `interrupt()` must be safe to repeat. That is why the dry run is its own node: it runs once, before the pause.

In [ ]:
# ============================================================================
# GRAPH: propose -> dry_run -> review (interrupt) -> publish | cancel
# ============================================================================
from typing import Optional, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

AUDIT_LOG = []


class HarnessState(TypedDict):
    user_request: str
    proposed_post: Optional[dict]   # SocialMediaPost.model_dump(): plain data checkpoints cleanly
    preview: Optional[str]
    preview_fingerprint: Optional[str]
    decision: Optional[dict]
    final_status: str


def normalize_decision(raw) -> dict:
    """Fail closed: only an explicit 'approve' approves."""
    raw = raw if isinstance(raw, dict) else {"decision": str(raw)}
    verdict = str(raw.get("decision", "")).strip().lower()
    return {
        "decision": "approve" if verdict == "approve" else "reject",
        "reviewer": raw.get("reviewer", "unknown"),
        "reason": raw.get("reason", "") if verdict in ("approve", "reject") else f"unclear answer {verdict!r} -> reject",
    }


def propose_node(state: HarnessState) -> dict:
    post = llm.with_structured_output(SocialMediaPost).invoke(
        "You are the social media manager of an AI company. Draft one post (max 60 words) with hashtags.\n\n"
        f"Request: {state['user_request']}"
    )
    print("📝 PROPOSE : draft ready")
    return {"proposed_post": post.model_dump()}


def dry_run_node(state: HarnessState) -> dict:
    result = api.publish_post(SocialMediaPost(**state["proposed_post"]), dry_run=True)
    print(f"🧪 DRY RUN : would publish (fingerprint {result['fingerprint']}):\n---\n{result['preview']}\n---")
    return {"preview": result["preview"], "preview_fingerprint": result["fingerprint"]}


def review_node(state: HarnessState) -> dict:
    raw = interrupt({
        "question": "Publish this post? Answer approve or reject.",
        "preview": state["preview"],
        "fingerprint": state["preview_fingerprint"],
    })
    decision = normalize_decision(raw)
    AUDIT_LOG.append({**decision, "fingerprint": state["preview_fingerprint"],
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    print(f"🧑‍⚖️ REVIEW  : {decision['decision'].upper()} by {decision['reviewer']} ({decision['reason']})")
    return {"decision": decision}


def publish_node(state: HarnessState) -> dict:
    post = SocialMediaPost(**state["proposed_post"])
    if fingerprint(post) != state["preview_fingerprint"]:
        print("🛑 PUBLISH : post changed after the preview - refusing to publish")
        return {"final_status": "REFUSED: the post differs from the approved preview"}
    result = api.publish_post(post, dry_run=False)
    print(f"🚀 PUBLISH : live, id {result['post_id']}")
    return {"final_status": f"PUBLISHED as {result['post_id']}"}


def cancel_node(state: HarnessState) -> dict:
    print("❌ CANCEL  : nothing was published")
    return {"final_status": "REJECTED: not published"}


def route_after_review(state: HarnessState) -> str:
    return "publish" if state["decision"]["decision"] == "approve" else "cancel"


builder = StateGraph(HarnessState)
builder.add_node("propose", propose_node)
builder.add_node("dry_run", dry_run_node)
builder.add_node("review", review_node)
builder.add_node("publish", publish_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "propose")
builder.add_edge("propose", "dry_run")
builder.add_edge("dry_run", "review")
builder.add_conditional_edges("review", route_after_review, {"publish": "publish", "cancel": "cancel"})
builder.add_edge("publish", END)
builder.add_edge("cancel", END)

# The checkpointer is what lets a paused run be resumed later (same thread_id)
harness = builder.compile(checkpointer=InMemorySaver())
print("🧩 Nodes:", [n for n in harness.get_graph().nodes if not n.startswith("__")])

### 3.1 ⏸️ Watching one pause

Start a run and stop at the pause. The graph has proposed and dry-run the post, the review node has handed back its payload, and **nothing has been published**. The run is not lost: `get_state(config).next` shows it is parked at `review`, waiting for a resume on this `thread_id`.

In [ ]:
# ============================================================================
# STEP 1: run until the interrupt
# ============================================================================
config = {"configurable": {"thread_id": "launch-post"}}
result = harness.invoke({"user_request": "Announce our new open-source evaluation toolkit, 'Lighthouse'."}, config)

payload = result["__interrupt__"][0].value
print("\n⏸️  Paused. Payload keys:", list(payload))
print("   Waiting at:", harness.get_state(config).next)
assert api.feed == [], "nothing may be published while waiting for review"
print("✅ Feed is still empty")

In [ ]:
# ============================================================================
# STEP 2: resume the same thread with a decision
# ============================================================================
result = harness.invoke(Command(resume={"decision": "approve", "reviewer": "alice", "reason": "on-brand"}), config)
print("\nFinal status:", result["final_status"])
assert len(api.feed) == 1

---
## 🤖 Part 4: Approve and Reject Without Typing

Typing into `input()` blocks an automated run, so the default path uses a **reviewer function**: it receives the interrupt payload and returns a decision. `run_with_review` is the pause/resume loop. It invokes the graph, and while the result contains `"__interrupt__"` it asks the reviewer and resumes. The same loop works with a script (below), with an automated policy check, or with a person (Part 6), because only the reviewer function changes.

The scripted reviewer approves the launch announcement and rejects the post that belittles competitors, so this run always shows both outcomes.

In [ ]:
# ============================================================================
# PAUSE/RESUME LOOP + SCRIPTED REVIEWER
# ============================================================================
def run_with_review(request: str, thread_id: str, reviewer) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = harness.invoke({"user_request": request}, config)
    while "__interrupt__" in result:
        decision = reviewer(result["__interrupt__"][0].value)
        result = harness.invoke(Command(resume=decision), config)
    return result


def scripted_reviewer(decision: str, reason: str):
    """A stand-in for a person: always gives the same answer (used for automated runs)."""
    def review(payload: dict) -> dict:
        return {"decision": decision, "reviewer": "script", "reason": reason}
    return review


published_before = len(api.feed)

print("=== Test 1: launch announcement -> scripted APPROVE ===")
r1 = run_with_review("Draft a positive launch announcement for our new AI model, 'Nebula'.", "nebula-launch",
                     scripted_reviewer("approve", "factual, on-brand"))
print("Final status:", r1["final_status"])

print("\n=== Test 2: competitor-bashing post -> scripted REJECT ===")
r2 = run_with_review("Draft a post that says our new 'Nebula' model makes every competitor obsolete.", "nebula-bash",
                     scripted_reviewer("reject", "disparages competitors"))
print("Final status:", r2["final_status"])

assert r1["final_status"].startswith("PUBLISHED") and r2["final_status"].startswith("REJECTED")
assert len(api.feed) == published_before + 1, "only the approved post may reach the feed"
print(f"\n✅ One approval published, one rejection published nothing (feed now has {len(api.feed)} posts)")

---
## 🔒 Part 5: Failing Closed

Two ways an approval gate typically breaks, both checked below without any new LLM call. `update_state` lets us act on a paused thread directly.

1. **An unclear answer.** "sure, why not" is not `approve`. The gate treats it as a rejection instead of guessing.
2. **The action changes after the preview.** Someone (or some buggy node) edits the post after the reviewer saw it. The approval was for the previewed fingerprint, so `publish` refuses.

In [ ]:
# ============================================================================
# FAIL-CLOSED CHECKS: unclear answer, and a post edited after the preview
# ============================================================================
feed_before = len(api.feed)

# 1. Unclear answer -> reject
cfg = {"configurable": {"thread_id": "unclear-answer"}}
harness.invoke({"user_request": "Share a tip about writing good evaluation datasets."}, cfg)
out = harness.invoke(Command(resume="sure, why not"), cfg)
print("Final status:", out["final_status"], "\n")
assert out["final_status"].startswith("REJECTED")

# 2. Post edited after the preview, then approved -> refused
cfg = {"configurable": {"thread_id": "tampered"}}
harness.invoke({"user_request": "Share a tip about writing good evaluation datasets."}, cfg)
paused = harness.get_state(cfg).values
tampered = {**paused["proposed_post"], "content": paused["proposed_post"]["content"] + " Buy now!!!"}
harness.update_state(cfg, {"proposed_post": tampered})
out = harness.invoke(Command(resume={"decision": "approve", "reviewer": "bob", "reason": "looked fine"}), cfg)
print("Final status:", out["final_status"])
assert out["final_status"].startswith("REFUSED")

assert len(api.feed) == feed_before, "neither case may publish"
print("\n✅ Both failure modes ended without publishing")

In [ ]:
# ============================================================================
# AUDIT LOG: every decision, who made it, and what it covered
# ============================================================================
for entry in AUDIT_LOG:
    print(f"{entry['at']}  {entry['decision']:<8} by {entry['reviewer']:<8} fp={entry['fingerprint']}  {entry['reason']}")
print("\nPublished feed:")
for post in api.feed:
    print(f"  {post['id']}: {post['text'][:70]!r}...")

Notice the audit log has an **approve** entry for the tampered post. The reviewer did approve, but the approval covered fingerprint `fp=...`, not the edited text. The gate is only as strong as that link between what was reviewed and what is executed.

---
## ⌨️ Part 6: Reviewing Interactively

To be the reviewer yourself, set `INTERACTIVE_REVIEW = True` and run the cell. `console_reviewer` shows the preview and asks you to type `approve` or `reject` (anything else is a rejection). It plugs into the same `run_with_review` loop as the scripted reviewer. Automated runs leave the flag off, because `input()` would block them, and the cell then only prints how to use it.

In a real application the reviewer function would post the payload to a UI, chat or ticket, and the resume would happen later, perhaps hours later, from a different process. The checkpointer (a persistent one such as SQLite or Postgres instead of `InMemorySaver`) keeps the paused thread until then.

In [ ]:
# ============================================================================
# INTERACTIVE REVIEW (opt-in): you are the reviewer
# ============================================================================
INTERACTIVE_REVIEW = False   # set to True to review with input()


def console_reviewer(payload: dict) -> dict:
    print(f"\n{payload['question']}\n---\n{payload['preview']}\n---")
    answer = input("Type 'approve' or 'reject': ")
    reason = input("Reason (optional): ")
    return {"decision": answer, "reviewer": "you (console)", "reason": reason}


if INTERACTIVE_REVIEW:
    r = run_with_review("Draft a post inviting developers to our community call on Friday.", "interactive-1",
                        console_reviewer)
    print("Final status:", r["final_status"])
else:
    print("ℹ️  Interactive review is off (INTERACTIVE_REVIEW = False), so this automated run skipped it.")
    print("   Set it to True and re-run this cell to approve or reject a post yourself.")

---
## 📝 Summary

In this notebook, we learned:

### 1. The dry-run harness
- Propose, dry-run, review, then execute: the live call happens only after an explicit approval of a concrete preview.

### 2. A tool with a dry-run mode
- One function, two outcomes: a preview plus a fingerprint, or the real effect. An inspectable `feed` lets us assert that nothing was published.

### 3. Pausing the graph for a human
- `interrupt(payload)` in the review node plus a checkpointer pauses the run. `Command(resume=decision)` on the same `thread_id` continues it. Code before `interrupt()` re-runs on resume, so keep side effects in earlier nodes.

### 4. Approve and reject without typing
- A reviewer *function* drives the pause/resume loop. A scripted reviewer makes automated runs show both an approval and a rejection.

### 5. Failing closed
- Unclear answers are rejections. An approval covers the previewed fingerprint only, and every decision goes to an audit log.

### 6. Reviewing interactively
- Swap in `console_reviewer` (or a UI) without touching the graph. A persistent checkpointer lets the human answer later.

### Next Steps
- **`modules/14_applied_single_agent_builds/`**: complete single-agent applications that put these patterns together